## Load OPENAI-API-KEY

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
from openai import OpenAI
import gabriel

# ---------- CONFIG ----------
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

# Load .env from proj_base (not the notebook's working directory)
dotenv_path = Path(proj_base)/".env"
print(dotenv_path)
load_dotenv(dotenv_path=dotenv_path)  # Explicitly specify the dotenv_path parameter

# Verify it loaded correctly (prints the actual key value to check if it's loaded)
api_key = os.environ.get("OPENAI_API_KEY")
print("API key loaded:", "Yes" if api_key else "No")

# Check if API key is available before creating the client
if not api_key:
    raise ValueError("OPENAI_API_KEY not found in environment variables. Check your .env file.")

# Use the key
client = OpenAI(api_key=api_key)



## Toy classification example.

If you followed the instructions above on setting up your API key, you should be able to press play on the cell below and see the magic happen! This is an essentially costless call. We encourage you to experiment with the data, labels, and parameters!

In [ ]:
import gabriel
from gabriel.utils.plot_utils import regression_plot, bar_plot, box_plot, line_plot
import os
import pandas as pd

### What we're doing here: we are creating a small toy spreadsheet, then passing it through the model to classify different features of each text.


# Part A. Create a dataframe. This is just like your own data, this line just creates a spreadsheet we can use to run through the package.
# this will work well with text from any language!
toy_data = pd.DataFrame({"text": [
    "I heard that the new president instituted a law on gun control.",
    "GDP REVAMP: WHAT CHANGED AND WHY",
    "Congress passes amended budget for 2026",
    "Why are you so late for class today? Did your dog eat your homework?",
    "Local activists rally outside city hall demanding climate reform.",
    "Scientists discover new species of frog in the Amazon rainforest.",
    "Breaking: Supreme Court rules on landmark case affecting digital privacy.",
    "I just finished reading a book that completely changed how I think about economics."
]})
# You can load real data using df = pd.read_csv("path_to_your_spreadsheet"). See the 'Loading your data' section for more info

# Part B. Define the classes we want to assess
labels = {
    "discusses politics": "The text makes reference to political figures, institutions, or issues.",
    "news headline": "The text is written in the style of a journalistic news headline.",
    "discusses animals": "The text includes mention of animals, whether real or metaphorical.",
    "first person perspective": "The text is written from a personal point of view.",
    "focus is on gun control": "The text’s primary subject is gun control, firearm regulations, or related debates."
}

# Part C. Make the call.
results = await gabriel.classify(
    toy_data,
    column_name = "text",       # name of column with text to classify
    labels = labels,            # dictionary of label definitions, defined above
    save_dir = "toy_classify",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5-mini",       # GPT model used for classification
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    n_runs = 1,                 # number of classification passes per text
    n_attributes_per_run = 8,   # if more than 8 labels, will split into separate calls of <= 8
    n_parallels = 650,          # max parallel threads (reduce if many errors, increase for higher speed)
    reset_files = False, # rerunning the cell loads from save / picks up from checkpoint
)

results     # press play on this cell to see your results

## Real Life Indian Newspaper Data
1. First attempt classification into [World, Sports, Sci/Tech, Business categories]

In [ ]:
from datasets import load_dataset 
news = load_dataset("roshuK7880/News-Classification-dataset")
news_headlines = news['train'].to_pandas().sample(60, random_state = 50)
news_headlines.head()

labels = {'world': 'Headline is a world/global news story',
          'sports': 'The article from the headline is about sports',
          'sci/tech': 'The headline is for an article about science or technology',
          'business': 'The article is about business',
          'politics': 'The article is about politics',
          'regulation': 'The article is about government regulation',
          'regional': 'The article is about a State or District or city in India'
         }

classifications = await gabriel.classify(
    df = news_headlines,
    column_name = 'text',
    labels = labels,
    model = 'gpt-5-nano',
    n_runs = 3,
    min_frequency = 0.3,          # minimum proportion of runs labeled "True" for the aggregate to be True
    save_dir = 'article_classification_labels',
    additional_instructions = "", # you could specify here that you only want the single best label
    reset_files = False,
)

In [ ]:
classifications.sample(20)
# you can use gabriel.view to read sample texts and their labels
gabriel.view(df = classifications, column_name = "text", attributes = labels, header_columns = ["label"])

# Rating attributes on text.

<mark>**We recommend starting here.**</mark> GABRIEL can rate texts on arbitrary features (e.g. `"populism"`) to measure numerical scores on consistent 0-100 scales.

While classification is great for subsetting big datasets and big picture understanding, it loses a lot of the nuance. It is just yes or no on a label, not "how much".

Rating attributes answers that "how much" question. With `gabriel.rate` specify some attributes you want to measure, and **get numbers on a 0-100 scale for each attribute, corresponding to how much the text manifests that attribute**. This preserves nuance, and is great for a regression.

For example, say you have a dataset of speeches. Your attributes might be `"populism"` or `"critical of the supreme court"` or `"adversarial"` or anything. You'll get a rating for each speech, on each attribute, 0 to 100. Same if you had interviews with people about their childhood upbringing: you could rate `"emphasizes family over friends"` or any other features of interest.

Run the toy example below. **We recommend you try out `gabriel.rank` too** - it also generates ratings, but by comparing the text to each other rather that rating them abstractly, making it more grounded.

Ratings are as simple as `results = await gabriel.rate(your_data, column_name, attributes_to_rate, save_folder)`!

In [ ]:

# Define the features we want to measure
attributes = {
    "factual": "How much the text centers on jobs, projects, deadlines, meetings, or professional goals (0 = not about work, 100 = entirely about work).",
    "optimism": "How positive and forward-looking the text feels (near 0 = pessimistic/hopeless, near 100 = highly hopeful and upbeat).",
    "anger": "How much frustration, irritation, or hostility is expressed (near 0 = none, 100 = very angry).",
    "neutral": "How formal, structured, and professional the language is (near 0 = very casual/slangy, near 100 = highly formal/academic).",
    "confidence": "How self-assured the article sounds about claims or outcomes (near 0 = doubtful/uncertain, near 100 = very certain/assertive).",
    "stressed": "How strained, overwhelmed, or anxious the article sounds (near 0 = calm/relaxed, near 100 = extremely stressed)."
}

additional_instructions = "" #### Any additional instructions you want to pass to the model when rating. For instance examples of how to rate the text

# Make the call to the package
results = await gabriel.rate(
    df = classifications,      # if using real data, substitute 'toy_data' with 'df'
    column_name = "text",       # name of column with text to rate
    attributes = attributes,    # attributes to score on 0–100 scale; defined above
    save_dir = "toy_rate",      # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5-mini",       # GPT model used for ratings
    modality = "text",  # input modality can also be "entity" (for terms like 'apple pie', not full texts), "web" (see web section), "pdf", "image", or "audio" (see multimodal section)
    ###
    ### the parameters below are less important
    ###
    additional_instructions = additional_instructions,
    n_runs = 1,                 # number of rating passes per text (higher = averaged over more ratings)
    n_attributes_per_run = 8,   # if more than 8 attributes, will split into separate calls of <= 8
    n_parallels = 650,          # max parallel threads (reduce if many errors, increase for higher speed)
    reasoning_effort = 'low',   # amount of reasoning effort, 'none' is default
    reset_files = False,  # if False, rerunning the cell loads from save / picks up from checkpoint
)

In [ ]:
results.sample(10)

In [ ]:
# you can use gabriel.view to read sample texts and their labels
gabriel.view(df = classifications, column_name = "text", attributes = labels, header_columns = ["label"])

## Rating entities (e.g. 'Andhra') instead of input text.

GPT already knows a great deal about India, Reliance, *Shah Bano v. Union Government*, etc. It doesn't need text to rate - we can take advantage of its internal knowledge and web search capacity.

Say you want to know which state is more business-friendly: Andhra or Bihar. These are both *entities*, not texts. Or you want to know which Diwali foods are the sweetest.

For many analyses, **acquiring text data is an unnecessary bottleneck; GPT's internal knowledge will suffice.**

You can pass entities (countries, companies, technologies, etc) via the `modality = "entity"` option. Everything else stays the same.


In [ ]:
toy_data = pd.DataFrame({"entity": [
    # States (28)
    "Andhra Pradesh",
    "Arunachal Pradesh",
    "Assam",
    "Bihar",
    "Chhattisgarh",
    "Goa",
    "Gujarat",
    "Haryana",
    "Himachal Pradesh",
    "Jharkhand",
    "Karnataka",
    "Kerala",
    "Madhya Pradesh",
    "Maharashtra",
    "Manipur",
    "Meghalaya",
    "Mizoram",
    "Nagaland",
    "Odisha",
    "Punjab",
    "Rajasthan",
    "Sikkim",
    "Tamil Nadu",
    "Telangana",
    "Tripura",
    "Uttar Pradesh",
    "Uttarakhand",
    "West Bengal",
    # Union territories (8)
    "Andaman and Nicobar Islands",
    "Chandigarh",
    "Dadra and Nagar Haveli and Daman and Diu",
    "Delhi",
    "Jammu and Kashmir",
    "Ladakh",
    "Lakshadweep",
    "Puducherry",
]})

attributes = {
    "Business friendly": "", # you can define an attribute in its value, or leave it blank
    "Restrictive to Business": "",  # defining attributes helps anchor every measurement on different entities to use the same 'rubric'
    "High GDP growth": "", # Internal assessment of GDP growth
}

results = await gabriel.rate(
    toy_data,
    column_name = "entity",          # name of column
    attributes = attributes,         # attributes to score
    save_dir = "entity_rate",
    model = "gpt-5-mini",            # GPT model used for scoring
    n_runs = 3,
    modality = "entity",
    reset_files = True,
)

In [ ]:
results

## Plot GPT entity (internal rating) information in a choropleth

In [ ]:
import plotly.graph_objs as go
import pandas as pd
import numpy as np
import requests
import io
import geopandas as gpd
import matplotlib.pyplot as plt
import os
from urllib.parse import urljoin

# Project folder
# proj_folder = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/Capex/"
proj_url = "https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/EPU"
# Poll Schedule files and folders 
base_url = 'https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/'
#State geo files
st_file_url = "https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/Capex/shape-files/3.cleaned/india_states_cleaned-V1.json"
#District files
dist_file_url = "https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/Capex/shape-files/1.raw/INDIA_DISTRICTS.geojson"
# Parliament Constituency files and folders 
pc_file_url = "https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/Capex/shape-files/1.raw/india_2014_parliament-V2.json"
# Assembly Constituency files and folders 
ac_file_url = "https://raw.githubusercontent.com/talktokalyan/Kalyan-Jupyter-Notebooks/main/Capex/shape-files/1.raw/india_2012-17_assembly-V2.json"

print('URLs declared for Geography Data')

try:
    response = requests.get(st_file_url, stream=True)
    response.raise_for_status()  # Raise HTTPError for bad responses (4xx or 5xx)

    # Use io.BytesIO to handle the file in memory
    excel_file = io.BytesIO(response.content)
    
    print(f"File download success")

except requests.exceptions.RequestException as e:
    print(f"Error downloading the file: {e}")
    exit()

# Read the GeoJSON file into a GeoDataFrame
states_gdf = gpd.read_file(st_file_url)

# # Group by 'State_Code' and 'State_Name' columns and extract unique combinations
# unique_states = states_gdf.groupby(['ST_CODE', 'ST_NAME']).size().reset_index(name='Count')

# # print("Unique State Codes and Names:")
# print(unique_states['ST_NAME'])

attributes = {
    "Business friendly": "", # you can define an attribute in its value, or leave it blank
    "Restrictive to Business": "",  # defining attributes helps anchor every measurement on different entities to use the same 'rubric'
    "GDP growth in 2024": "", # Internal assessment of GDP growth
}
results = await gabriel.rate(
    states_gdf,
    column_name = "ST_NAME",          # name of column
    attributes = attributes,         # attributes to score
    save_dir = "entity_rate",
    model = "gpt-5-mini",            # GPT model used for scoring
    n_runs = 3,
    modality = "entity",
    reset_files = True,
)

print(results)

# Plot the choropleth map
fig, ax = plt.subplots(figsize=(15, 15))

# Set the background color of the figure and axes
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Plot the choropleth map
results.plot(ax=ax, column='GDP growth in 2024', legend=True, cmap='Accent', edgecolor='black', linewidth=0.5)
# Plot the state borders with a thicker line
states_gdf.boundary.plot(ax=ax, edgecolor='black', linewidth=1)

# Remove the axis for a cleaner look
ax.axis('off')

# Add a title
plt.title('GPT rating of "GDP growth in 2024"')

# Display the plot
plt.show()




## Load data from pdf and extract attributes using GPT prompts
### Replication of Baker and Bloom (2016)

In [ ]:
import matplotlib.pyplot as plt
import os
import PyPDF2
import gabriel
from urllib.parse import urljoin
from openai import OpenAI

# Placeholder for folder and file name
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

folder = 'data/pdfs/'  # Added trailing slash to ensure proper path joining
filename = 'BS_Mumbai_26-11-2025.pdf'

# Join to get full path to the PDF file
pdf_file = os.path.join(proj_base, folder, filename)  # Properly join path components

# Check if the file exists before attempting to load it
if not os.path.exists(pdf_file):
    print(f"File not found: {pdf_file}")
else:
    print(f"File found: {pdf_file}")
    
    # Extracting from a PDF
    # Make sure gabriel is imported if you're using it
    # import gabriel  # Uncomment if needed
    
pdf_df = gabriel.load(pdf_file, modality="pdf", reset_files=True)
    
print(pdf_df.columns)

attributes = {
    "date": "Extract Date of the newspaper in dd/mm/yyyy date format",
    "headline": "News headline",
    "text": "Extract full text of the news story only and ignore subtitle, author name and date",
    "EPU": "Set value 1 if the text of each separate news article contains at least one term from each of the three term sets. The first set is uncertain, uncertainties, or uncertainty. The second set is 'economic' or 'economy'. The third set consists of policy relevant terms such as 'regulation', 'central bank', 'monetary policy', 'policymakers', 'deficit', 'legislation', and 'fiscal policy'."
    
}

# will return one row per request (hundreds of rows per pdf)
extract_results = await gabriel.extract(
    df = pdf_df,
    column_name = "path",      # name of column with content to extract from
    attributes = attributes,   # attributes to extract; returns "<NA>" if GPT doesn't know
    additional_instructions=(
        "Treat each distinct news story as a separate article. "
        "Ignore ads and stock tables. If an article continues on another page, "
        "Include only the visible part on this page."
        "Go through entire pdf and extract all news stories"
    ),
    save_dir = "toy_pdf_extract",  # directory to save results, use a Google Drive folder (e.g. "/content/drive/folder") for permanent storage (see 'Loading your data' section)
    model = "gpt-5.2",
    modality = "pdf",
    reasoning_effort = "medium",
    reset_files = True,
)


In [ ]:
import os
import glob
import pandas as pd
from openai import OpenAI
import gabriel

# ---------- CONFIG ----------
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"
pdf_folder = os.path.join(proj_base, "data/pdfs")
output_folder = os.path.join(proj_base, "output")
os.makedirs(output_folder, exist_ok=True)

output_csv = os.path.join(output_folder, "ALL_EPU_classified.csv")

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ---------- NEWSPAPER SHORTCODE MAPPING ----------

newspaper_instructions = """
NEWSPAPER IDENTIFICATION RULES:
Return exactly one 3-letter shortcode from this list based on the newspaper name/logo:
- BSD = Business Standard
- ET  = The Economic Times
- TOI = Times of India
- HT  = Hindustan Times  
- HIN = The Hindu
- STA = The Statesman
- IEX = Indian Express
- FE  = Financial Express
- MNT = MINT

Look for the masthead, logo, or publication name at the top of pages.
"""

# ---------- GABRIEL ATTRIBUTES ----------

attributes = {
    "newspaper": (
        "Identify the newspaper name and return ONLY its 3-letter shortcode. "
        f"{newspaper_instructions}"
    ),
    "date": (
        "Extract the date of the newspaper in dd/mm/yyyy format. "
        "Look for the publication date on the front page."
    ),
    "headline": (
        "The main news headline of a single article, copied verbatim from the newspaper. "
        "Extract only the primary headline, not subheadlines or author names."
    ),
    "text": (
        "Extract the full text of the news story only. "
        "Ignore subtitle, author name, location, date, advertisements, stock tables, "
        "and page references like 'Turn to Page X'. "
        "Stop at the point where a new article clearly begins."
    ),
    "EPU": (
        "Set value to 1 if the text of this news article contains at least one term "
        "from EACH of the three term sets below. Set value to 0 if any term set is missing. "
        "\n\nTerm Set 1 (Uncertainty): 'uncertain', 'uncertainties', or 'uncertainty'"
        "\n\nTerm Set 2 (Economic): 'economic' or 'economy'"
        "\n\nTerm Set 3 (Policy): 'regulation', 'central bank', 'monetary policy', "
        "'policymakers', 'deficit', 'legislation', 'fiscal policy', "
        "'RBI', 'Reserve Bank', 'government policy', 'Budget', 'tax policy', "
        "'subsidy', 'tariff', or 'trade policy'"
        "\n\nReturn only '1' or '0'."
    ),
"State": ( " Identify if the news article states strictly something about Economic Policy Uncertainty or Regulation particular to a state or union territory of India." 
          "Return the two letter codes of the Indian State or Union Territory"
          "If multiple states are mentioned. Limit the State codes to maximum of three, by separating them with a comma punctuation"
         ),  
}

additional_instructions = (
    "Go through the entire PDF and extract ALL news stories, no matter how small. "
    "Treat each distinct news story as a separate article. "
    "Each article has a headline followed by body text. "
    "Ignore images, advertisements, stock market tables, and promotional content. "
    "If an article is truncated with 'Turn to Page X', include only the visible text on this page. "
    "For EPU classification, carefully check if the article text contains at least one term "
    "from ALL THREE term sets."
)

# ---------- LOOP OVER ALL PDFs ----------

all_results = []

pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files in {pdf_folder}")

for pdf_file in pdf_files:
    print(f"\nProcessing file: {pdf_file}")

    if not os.path.exists(pdf_file):
        print(f"  Skipping, file not found.")
        continue

    # 1) Load PDF into Gabriel
    try:
        pdf_df = gabriel.load(pdf_file, modality="pdf", reset_files=True)
    except Exception as e:
        print(f"  Error loading PDF: {e}")
        continue

    # 2) Extract articles + EPU classification + newspaper
    try:
        extract_results = await gabriel.extract(
            df=pdf_df,
            column_name="path",
            attributes=attributes,
            additional_instructions=additional_instructions,
            save_dir="epu_pdf_extract",
            model="gpt-5.2",
            modality="pdf",
            reasoning_effort="high",
            reset_files=True,
        )
    except Exception as e:
        print(f"  Error during extraction: {e}")
        continue

    # 3) Convert to DataFrame and add file info
    df_file = pd.DataFrame(extract_results)

    print(f"Extracted {len(df_file)} articles from {os.path.basename(pdf_file)}")

    all_results.append(df_file)

# ---------- COMBINE ALL RESULTS ----------

if not all_results:
    print("No data extracted from any PDF.")
else:
    combined_df = pd.concat(all_results, ignore_index=True)

    # ---------- CLEANUP: Remove path and date_parsed ----------

    # Remove path column (if it exists)
    if "path" in combined_df.columns:
        combined_df = combined_df.drop(columns=["path"])

    # ---------- ADD PER-DAY ARTICLE COUNT (without date_parsed column) ----------

    # Use the original 'date' column directly for counting
    counts = (
        combined_df
        .dropna(subset=["date", "headline"])
        .groupby("date")["headline"]
        .nunique()
        .rename("stories_per_day")
    )

    combined_df = combined_df.merge(
        counts,
        how="left",
        left_on="date",
        right_index=True,
    )

    # ---------- SAVE SINGLE CSV ----------

    combined_df.to_csv(output_csv, index=False)
    print(f"\nCombined results saved to: {output_csv}")
    print(f"Total articles across all PDFs: {len(combined_df)}")
    print("Final columns:", list(combined_df.columns))

    # ---------- NEWSPAPER SUMMARY ----------

    print("\n" + "="*60)
    print("NEWSPAPER SUMMARY")
    print("="*60)
    newspaper_counts = combined_df["newspaper"].value_counts()
    for code, count in newspaper_counts.items():
        print(f"{code}: {count} articles")

    # ---------- EPU SUMMARY BY NEWSPAPER ----------

    print("\n" + "="*60)
    print("EPU BY NEWSPAPER")
    print("="*60)
    epus = combined_df.groupby(["newspaper", "EPU"]).size().unstack(fill_value=0)
    print(epus)


### State + General EPU
POC: 21 March 2026 
1. Code to extract Economic Sentiment using OPEN AI GABRIEL using Natural language prompts.
2. State Attributes for articles extracted as two-letter codes
3. Flags State Economic Policy uncertainty (1(Yes)/ 0 (No)).
4. Files output to excel for comparison with sentiment generated by VADER package

In [6]:
import os
import glob
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import gabriel
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


# ---------- CONFIG ----------
proj_base = "/Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/"

# Load .env from proj_base (not the notebook's working directory)
dotenv_path = Path(proj_base) / ".env"
load_dotenv(dotenv_path)

# Verify it loaded correctly (prints True if found, False if not)
print("API key loaded:", bool(os.environ.get("OPENAI_API_KEY")))
# Use the key
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# ---------- FILE INPUT AND OUTPUT PATH ----------

pdf_folder    = os.path.join(proj_base, "data/pdfs copy")
output_folder = os.path.join(proj_base, "output")
os.makedirs(output_folder, exist_ok=True)
output_csv    = os.path.join(output_folder, "ALL_EPU_classified-2.csv")

newspaper_instructions = """
NEWSPAPER IDENTIFICATION RULES:
Return exactly one 3-letter shortcode from this list based on the newspaper name/logo:
- BSD = Business Standard
- ET  = The Economic Times
- TOI = Times of India
- HT  = Hindustan Times  
- HIN = The Hindu
- STA = The Statesman
- IEX = Indian Express
- FE  = Financial Express
- MNT = MINT

Look for the masthead, logo, or publication name at the top of pages.
"""

# ---------- GABRIEL ATTRIBUTES ----------

attributes = {
    "newspaper": (
        "Identify the newspaper name and return ONLY its 3-letter shortcode. "
        f"{newspaper_instructions}"
    ),
    "date": (
        "Extract the date of the newspaper in dd/mm/yyyy format. "
        "Look for the publication date on the masthead of the front page."
    ),

    "headline": (
        "Primary headline of this article only. "
        "Verbatim. No subheadlines, no author names."
    ),

    # --- FULL TEXT (for offline local sentiment packages) ---
    "text": (
        "Extract the full text of this news article only. "
        "Exclude: headline, sub-headline, author name, dateline, captions, ads, "
        "Exclude: Infographics, images, charts, stock tables, and 'Turn to Page X' references. "
        "Stop when the next article begins."
    ),

    # --- GABRIEL SENTIMENT SCORE ---
    "gabriel_econ_sentiment": (
        "Rate the economic sentiment of this article on a continuous scale 0–100. "
        "0 = extremely negative (crisis, collapse, recession, contraction). "
        "50 = neutral or mixed economic outlook. "
        "100 = extremely positive (boom, strong growth, recovery, expansion). "
        "Consider only economic and policy content, not general tone. "
    ),

    # --- EPU FLAG ---
    "EPU": (
        "Ignore case of the text of the news article."
        "Return 1 if the article contains at least one word from ALL THREE sets, else 0. "
        "Set1-Uncertainty: uncertain/uncertainties/uncertainty. "
        "Set2-Economy: economic/economy. "
        "Set3-Policy: regulation/central bank/monetary policy/policymakers/"
        "deficit/legislation/fiscal policy/RBI/Reserve Bank/"
        "government policy/Budget/tax policy/subsidy/tariff/trade policy. "
        "Return only: 1 or 0."
    ),

    # --- STATE / UT FLAG ---
    "state_flag": (
        "Is the news article specific to any Indian state or Union Territory? "
        "Does this article discuss uncertainty or ambiguity regarding economic policy or regulation"
        "Return 1 if yes, 0 if national/international."
    ),

    "state_codes": (
        "If state_flag=1, return up to 3 Indian state/UT vehicle registration codes. "
        "Examples: AP, MH, KA, TN, DL, GJ, RJ, UP, WB, TS, KL, PB, HR, OD, BR, MP, "
        "CG, JH, UK, HP, GA, AS, MN, MZ, NL, TR, AR, SK, ML, JK, LA, "
        "CH, PY, AN, DN, DD, LD. "
        "List dominant state first. Separate with commas. "
        "Return NA if state_flag=0."
    ),
}

# ---------- ADDITIONAL INSTRUCTIONS ----------
additional_instructions = (
    "Extract every distinct news article from the full PDF as a separate row. "
    "Skip ads, stock tables, weather, and TV schedules. "
    "For truncated articles ('Turn to Page X'), use only the visible portion. "
    "EPU check must use the full article text, not just the headline."
)

# ---------- LOOP OVER ALL PDFs ----------
all_results = []
pdf_files = sorted(glob.glob(os.path.join(pdf_folder, "*.pdf")))
print(f"Found {len(pdf_files)} PDF files\n")

for pdf_file in pdf_files:
    fname = os.path.basename(pdf_file)
    print(f"Processing: {fname}")

    # 1) Load PDF
    try:
        pdf_df = gabriel.load(pdf_file, modality="pdf", reset_files=True)
    except Exception as e:
        print(f"  Load error: {e}")
        continue

    # 2) Extract via Gabriel
    try:
        extract_results = await gabriel.extract(
            df                      = pdf_df,
            column_name             = "path",
            attributes              = attributes,
            additional_instructions = additional_instructions,
            save_dir                = "epu_pdf_extract",
            model                   = "gpt-5.4",
            modality                = "pdf",
            reasoning_effort        = "high",
            reset_files             = True,
        )
    except Exception as e:
        print(f"  Extract error: {e}")
        continue

    # 3) To DataFrame
    df_file = pd.DataFrame(extract_results)
    df_file["source_file"] = fname
    print(f"  → {len(df_file)} articles extracted")
    all_results.append(df_file)

# ---------- COMBINE ----------
if not all_results:
    print("No data extracted.")
else:
    combined_df = pd.concat(all_results, ignore_index=True)

    # --- Drop all internal Gabriel columns ---
    gabriel_cols = ["path", "entity_name", "entity_id"]
    combined_df = combined_df.drop(
        columns=[c for c in gabriel_cols if c in combined_df.columns]
    )

    # --- Keep only intended columns ---
    intended_cols = list(attributes.keys()) + ["source_file"]
    combined_df = combined_df[[c for c in intended_cols if c in combined_df.columns]]

    # --- Convert numeric types ---
    combined_df["EPU"]               = pd.to_numeric(combined_df["EPU"],               errors="coerce")
    combined_df["state_flag"]        = pd.to_numeric(combined_df["state_flag"],        errors="coerce")
    combined_df["gabriel_econ_sentiment"] = pd.to_numeric(combined_df["gabriel_econ_sentiment"], errors="coerce")

    # --- Per-day story count ---
    counts = (
        combined_df
        .dropna(subset=["date", "newspaper", "headline"])
        .groupby(["date", "newspaper"], as_index=False)["headline"]
        .nunique()
        .rename(columns={"headline": "num_art_np_day"})
    )

    combined_df = combined_df.merge(
        counts, on=["date", "newspaper"], how="left"
    )
    # --------------------------------------------------------
    # LOCAL SENTIMENT SCORING (zero API cost, uses 'text' column)
    # --------------------------------------------------------

    # --- VADER score (0–100) ---
    # pip install vaderSentiment
    vader = SentimentIntensityAnalyzer()

    def vader_score(text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return None
        raw = vader.polarity_scores(text)["compound"]  # -1 to +1
        return round((raw + 1) * 50, 1)                # rescale to 0–100

    combined_df["vader_sentiment"] = combined_df["text"].apply(vader_score)

    # --- TextBlob score (0–100) ---
    # pip install textblob
    from textblob import TextBlob

    def textblob_score(text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return None
        raw = TextBlob(text).sentiment.polarity        # -1 to +1
        return round((raw + 1) * 50, 1)               # rescale to 0–100

    combined_df["textblob_sentiment"] = combined_df["text"].apply(textblob_score)

    # --- FinBERT score (0–100) — best for financial/economic news ---
    # pip install transformers torch
    # Uncomment the block below when ready; it is slower but most accurate
    
    from transformers import pipeline
    finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert")
    
    def finbert_score(text):
        if not isinstance(text, str) or len(text.strip()) == 0:
            return None
        result = finbert(text[:512])[0]
        label  = result["label"]     # positive / negative / neutral
        conf   = result["score"]     # 0 to 1
        if label == "positive":
            return round(50 + conf * 50, 1)   # 50–100
        elif label == "negative":
            return round(50 - conf * 50, 1)   # 0–50
        else:
            return 50.0                        # neutral
    
    combined_df["finbert_sentiment"] = combined_df["text"].apply(finbert_score)

    # --- Save ---
    combined_df.to_csv(output_csv, index=False)
    print(f"\nSaved {len(combined_df)} articles → {output_csv}")
    print("Columns:", list(combined_df.columns))

    # --- Summaries ---
    print("\n" + "="*60)
    print("NEWSPAPER SUMMARY")
    print("="*60)
    print(combined_df["newspaper"].value_counts().to_string())

    print("\n" + "="*60)
    print("EPU BY NEWSPAPER")
    print("="*60)
    print(combined_df.groupby(["newspaper","EPU"]).size().unstack(fill_value=0).to_string())

    print("\n" + "="*60)
    print("SENTIMENT COMPARISON BY NEWSPAPER (mean 0–100)")
    print("="*60)
    sent_cols = ["gabriel_econ_sentiment", "vader_sentiment", "textblob_sentiment"]
    print(combined_df.groupby("newspaper")[sent_cols].mean().round(1).to_string())

    print("\n" + "="*60)
    print("STATE-LEVEL ARTICLES")
    print("="*60)
    state_df = combined_df[combined_df["state_flag"] == 1]
    print(f"State-specific articles: {len(state_df)}")
    state_series = (
        state_df["state_codes"]
        .dropna()
        .str.split(",")
        .explode()
        .str.strip()
        .value_counts()
    )
    print(state_series.head(10).to_string())


API key loaded: True
Found 6 PDF files

Processing: ET-Bangalore 06-04.pdf
[gabriel.load] PDF modality attaches PDFs directly (richer layout, figures, and images). Set modality='text' (or 'entity'/'web') to extract text-only versions of PDFs.
                     name                                               path
0  ET-Bangalore 06-04.pdf  /Users/kalyan/Library/CloudStorage/OneDrive-Pe...
Saved aggregated file to /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/data/pdfs copy/gabriel_aggregated_content.csv
[Extract] Rendering 1 prompts…
Initializing model calls and loading data...

===== Run kickoff =====
Prompts: 1 | Words: ~905 | Words per prompt: ~905
Model: gpt-5.4 | Reasoning effort: high | Mode: streaming | modality: pdf
Pricing for model 'gpt-5.4': input $2.5/1M, output $15.0/1M
Estimated token usage: input 3,357, output 500 | ~4,357 tokens per call
Estimated synchronous cost: $0.02 (input: $0.01, output: $0.01)
Note: multim

Processing prompts:   0%|                                   | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-04-08 10:48:12 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=37, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 36 to 180 parallel threads over 15s.
[parallelization] Ramp-up complete at 180 parallel threads.
2026-04-08 10:50:12 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
2026-04-08 10:52:12 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
2026-04-

Processing prompts:   0%|                                   | 0/1 [00:00<?, ?it/s]

[parallelization] 2026-04-08 11:14:55 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=37, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 36 to 180 parallel threads over 15s.
[parallelization] Ramp-up complete at 180 parallel threads.
2026-04-08 11:16:55 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
2026-04-08 11:18:55 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
[dynamic

...

[parallelization] 2026-04-08 12:10:09 | Initial parallelization settings: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=37, active=0, inflight=0, awaiting_response=0, queue=1, processed=0/1
[parallelization] Ramping up from 36 to 180 parallel threads over 15s.
[parallelization] Ramp-up complete at 180 parallel threads.
2026-04-08 12:12:09 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
2026-04-08 12:14:09 | Periodic status update: cost_so_far=$0.00, p90=unknown, timeouts=0/1, rate_limit_errors=0/1, connection_errors=0/1, json_parse_errors=0/1, tps=unknown, throughput<=180 prompts/min, cap=180, active=1, inflight=1, awaiting_response=1, queue=0, processed=0/1
2026-04-

The following layers were not sharded: bert.embeddings.position_embeddings.weight, bert.encoder.layer.*.attention.self.key.bias, bert.encoder.layer.*.attention.self.key.weight, bert.encoder.layer.*.attention.self.value.bias, bert.encoder.layer.*.attention.output.dense.bias, bert.encoder.layer.*.attention.self.value.weight, bert.encoder.layer.*.output.LayerNorm.weight, bert.encoder.layer.*.output.dense.weight, bert.encoder.layer.*.attention.output.LayerNorm.bias, bert.encoder.layer.*.output.dense.bias, classifier.weight, bert.encoder.layer.*.output.LayerNorm.bias, bert.embeddings.word_embeddings.weight, bert.encoder.layer.*.attention.output.dense.weight, classifier.bias, bert.encoder.layer.*.intermediate.dense.weight, bert.pooler.dense.bias, bert.encoder.layer.*.intermediate.dense.bias, bert.encoder.layer.*.attention.output.LayerNorm.weight, bert.embeddings.token_type_embeddings.weight, bert.pooler.dense.weight, bert.encoder.layer.*.attention.self.query.bias, bert.encoder.layer.*.attent

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Saved 334 articles → /Users/kalyan/Library/CloudStorage/OneDrive-Personal/Kalyan/KK-Python/Kalyan-Jupyter-Notebooks/EPU/output/ALL_EPU_classified-2.csv
Columns: ['newspaper', 'date', 'headline', 'text', 'gabriel_econ_sentiment', 'EPU', 'state_flag', 'state_codes', 'source_file', 'num_art_np_day', 'vader_sentiment', 'textblob_sentiment', 'finbert_sentiment']

NEWSPAPER SUMMARY
newspaper
ET     186
MNT    148

EPU BY NEWSPAPER
EPU          0  1
newspaper        
ET         186  0
MNT        146  2

SENTIMENT COMPARISON BY NEWSPAPER (mean 0–100)
           gabriel_econ_sentiment  vader_sentiment  textblob_sentiment
newspaper                                                             
ET                           50.5             57.7                53.2
MNT                          49.1             59.2                52.9

STATE-LEVEL ARTICLES
State-specific articles: 42
state_codes
KL    9
TN    8
AS    7
WB    6
PY    4
KA    3
DL    3
MN    2
MH    2
LA    1


## Cost effective EPU extraction (English)
Still a mystery